<a href="https://colab.research.google.com/github/ChadapaNgamchuen/LLM-Economic-News-Summarizer/blob/main/Economic_News_AI_Summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Summary (AI-news-summarizer)

This notebook demonstrates how DSPy can structure LLM outputs into
reliable, typed fields — without manually writing prompt templates.

Key techniques used:
  - DSPy Signature  : Defines input/output schema for the AI task
  - ChainOfThought  : Encourages step-by-step reasoning before answering
  - Batch loop      : Processes multiple articles in sequence
  - pandas styling  : Formats results into a readable, filterable table

Potential next steps:
  - Add more news sources or connect to a live RSS feed
  - Use DSPy Optimizer to automatically improve prompt quality

In [ ]:
!pip install dspy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.4/312.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 14.2 MB/s eta 0:00:00


In [ ]:
import dspy
import pandas as pd
import os

lm = dspy.LM("gemini/gemini-2.5-flash", api_key="")
dspy.configure(lm=lm)

class NewsSummarizer(dspy.Signature):
    """You are an expert in economics. Summarize the news clearly and give insights."""
    news_text: str     = dspy.InputField(desc="News article text")
    summary: str       = dspy.OutputField(desc="Short summary (3-5 lines)")
    bullet_points: str = dspy.OutputField(
        desc="Key points, each on a NEW LINE starting with *. "
             "Format exactly like this:\n* point one\n* point two\n* point three"
    )
    insight: str       = dspy.OutputField(desc="Important insight or impact")
    sentiment: str     = dspy.OutputField(desc="Sentiment: Positive / Negative / Neutral")
    tags: str          = dspy.OutputField(
        desc="2-3 category tags as comma-separated list. "
             "Choose from: Monetary Policy, Central Bank, Inflation, "
             "Trade, Exports, Commodities, Energy, Safe-haven, "
             "Consumer, Real Estate, Currency, Stock Market, Crypto. "
    )

summarizer = dspy.ChainOfThought(NewsSummarizer)

def clean_bullets(text: str) -> str:

    parts = [p.strip() for p in text.split("*") if p.strip()]
    return "* " + "\n* ".join(parts)

def summarize_news(news_text: str) -> dict:
    print(f"กำลังวิเคราะห์: {news_text[:50]}...")
    result = summarizer(news_text=news_text)

    return {
        "News":       news_text,
        "Summary":    getattr(result, 'summary', ''),
        "Key Points": clean_bullets(getattr(result, 'bullet_points', '')),
        "Insight":    getattr(result, 'insight', ''),
        "Sentiment":  getattr(result, 'sentiment', ''),
        "Tags":       getattr(result, 'tags', '')
    }

news_list = [
    "Thailand's central bank decided to keep interest rates unchanged due to stable inflation and moderate economic growth.",
    "Gold prices hit record high amid global uncertainty as investors seek safe-haven assets.",
    "Thailand's export sector contracted for the third consecutive month, driven by weak demand from China.",
    "OPEC signals production increase, causing crude oil prices to drop 5% this week.",
    "Thailand faces rising energy costs due to global oil supply disruptions, raising concerns about inflation."
]

all_news_data = []
for news in news_list:
    result_dict = summarize_news(news)
    all_news_data.append(result_dict)


df = pd.DataFrame(all_news_data)
df = df[["News", "Summary", "Key Points", "Insight", "Sentiment", "Tags"]]

styled_df = df.style.set_properties(**{
    'white-space': 'pre-wrap',
    'text-align': 'left',
    'vertical-align': 'top'
})

styled_df

กำลังวิเคราะห์: Thailand's central bank decided to keep interest r...
กำลังวิเคราะห์: Gold prices hit record high amid global uncertaint...
กำลังวิเคราะห์: Thailand's export sector contracted for the third ...
กำลังวิเคราะห์: OPEC signals production increase, causing crude oi...
กำลังวิเคราะห์: Thailand faces rising energy costs due to global o...


,News,Summary,Key Points,Insight,Sentiment,Tags
0,Thailand's central bank decided to keep interest rates unchanged due to stable inflation and moderate economic growth.,"Thailand's central bank opted to keep its benchmark interest rate unchanged, citing a period of stable inflation and steady, moderate economic growth. This decision reflects the central bank's view that current monetary policy is adequately supporting the economy without fueling price instability.",* Thailand's central bank decided to maintain interest rates. * The decision was driven by stable inflation levels. * Moderate economic growth was another key factor in the decision. * The central bank sees no immediate need to adjust policy.,"The central bank's decision signals confidence in the current economic trajectory and stability. It suggests that policymakers believe the economy is growing at a sustainable pace without overheating or facing significant deflationary risks, allowing for a continuation of the current monetary stance to support long-term stability.",Neutral,"Monetary Policy, Central Bank, Inflation"
1,Gold prices hit record high amid global uncertainty as investors seek safe-haven assets.,"Gold prices have reached an unprecedented high as investors flock to the precious metal, viewing it as a safe-haven asset amidst prevalent global uncertainty. This surge indicates a broad market sentiment of caution, with capital being reallocated from riskier investments to more stable stores of value.",* Gold prices have climbed to a new record high. * The increase is attributed to widespread global uncertainty. * Investors are actively seeking safe-haven assets to protect their capital.,"The record high in gold prices signals a significant level of investor apprehension about the global economic and geopolitical outlook. It suggests that market participants are prioritizing capital preservation over growth, indicating potential headwinds for riskier assets and a possible slowdown in economic activity as uncertainty persists.",Neutral,"Safe-haven, Commodities, Stock Market"
2,"Thailand's export sector contracted for the third consecutive month, driven by weak demand from China.","Thailand's export sector has faced a third consecutive month of contraction, primarily attributed to a downturn in demand from China. This sustained decline signals ongoing challenges for Thailand's trade performance and its economic reliance on major trading partners.",* Thailand's export sector contracted for the third straight month. * The contraction is driven by weak demand from China.,"A prolonged contraction in exports, especially due to reduced demand from a significant trading partner like China, is a negative indicator for Thailand's economic health. It suggests a potential slowdown in global or regional trade, which could lead to reduced manufacturing output, potential job losses, and pressure on the national currency. This also underscores the vulnerability of export-dependent economies to external demand shocks.",Negative,"Exports, Trade"
3,"OPEC signals production increase, causing crude oil prices to drop 5% this week.","OPEC's recent signaling of increased oil production has directly caused a significant 5% drop in crude oil prices this week. This move indicates a higher supply outlook for the global market, consequently driving down the commodity's value. The market reacted swiftly to the prospect of more oil becoming available.",* OPEC has indicated an intention to increase oil production. * Crude oil prices have subsequently fallen by 5% this week. * The price drop is a direct market reaction to the anticipated increase in global oil supply.,"This significant drop in crude oil prices, driven by OPEC's production increase, can have a dual impact on the global economy. While it offers potential relief for consumers and businesses through lower energy costs, possibly easing inflationary pressures, it simultaneously poses financial challenges for oil-producing nation

# AI News Summarizer

This project summarizes economic news using Gemini API.

## Step 1: Setup Model

import the required libraries and configure the language model.
- dspy: A framework for programming language models in a structured way
- pandas: For organizing results into a table
- dspy.LM: Connects to Gemini 2.5 Flash as our AI engine
- dspy.configure: Sets Gemini as the default model for all DSPy modules

In [ ]:
import dspy
import pandas as pd
import os

lm = dspy.LM("gemini/gemini-2.5-flash", api_key="key")
dspy.configure(lm=lm)

## Step 2: Define the Task

A DSPy Signature acts as a "contract" — it tells the AI exactly what input to expect
and what outputs to return.

Input:
  - news_text: The raw news article

Outputs:
  - summary     : A concise 3–5 line overview of the news
  - bullet_points: Key facts formatted as bullet points
  - insight     : Economic impact or implication of the news
  - sentiment   : Overall tone — Positive, Negative, or Neutral
  - tags        : 2–3 category labels for filtering (e.g., Inflation, Trade)

In [ ]:
class NewsSummarizer(dspy.Signature):
    """You are an expert in economics. Summarize the news clearly and give insights."""
    news_text: str     = dspy.InputField(desc="News article text")
    summary: str       = dspy.OutputField(desc="Short summary (3-5 lines)")
    bullet_points: str = dspy.OutputField(
        desc="Key points, each on a NEW LINE starting with *. "
             "Format exactly like this:\n* point one\n* point two\n* point three"
    )
    insight: str       = dspy.OutputField(desc="Important insight or impact")
    sentiment: str     = dspy.OutputField(desc="Sentiment: Positive / Negative / Neutral")
    tags: str          = dspy.OutputField(
        desc="2-3 category tags as comma-separated list. "
             "Choose from: Monetary Policy, Central Bank, Inflation, "
             "Trade, Exports, Commodities, Energy, Safe-haven, "
             "Consumer, Real Estate, Currency, Stock Market, Crypto. "
    )

## Step 3: Define the Task

dspy.ChainOfThought wraps the Signature and instructs the model to reason
step-by-step before producing each output field. This improves accuracy
compared to asking for all outputs directly.

clean_bullets() is a helper function that reformats the bullet_points string
so each point appears on its own line, improving readability in the table.

summarize_news() calls the model for a single article and returns a
dictionary with all output fields ready to be added to our dataset.

In [ ]:
summarizer = dspy.ChainOfThought(NewsSummarizer)

def clean_bullets(text: str) -> str:

    parts = [p.strip() for p in text.split("*") if p.strip()]
    return "* " + "\n* ".join(parts)

def summarize_news(news_text: str) -> dict:
    print(f"กำลังวิเคราะห์: {news_text[:50]}...")
    result = summarizer(news_text=news_text)

    return {
        "News":       news_text,
        "Summary":    getattr(result, 'summary', ''),
        "Key Points": clean_bullets(getattr(result, 'bullet_points', '')),
        "Insight":    getattr(result, 'insight', ''),
        "Sentiment":  getattr(result, 'sentiment', ''),
        "Tags":       getattr(result, 'tags', '')
    }

## Step 4: Define the Task

Define a list of five economic news headlines covering different topics:
  1. Thailand central bank interest rate decision
  2. Global gold price surge
  3. Thailand export contraction
  4. OPEC oil production announcement
  5. Thailand energy cost impact

These represent a realistic mix of Monetary Policy, Commodities, Trade,
and Energy news — useful for testing the tag classification system.

In [ ]:
news_list = [
    "Thailand's central bank decided to keep interest rates unchanged due to stable inflation and moderate economic growth.",
    "Gold prices hit record high amid global uncertainty as investors seek safe-haven assets.",
    "Thailand's export sector contracted for the third consecutive month, driven by weak demand from China.",
    "OPEC signals production increase, causing crude oil prices to drop 5% this week.",
    "Thailand faces rising energy costs due to global oil supply disruptions, raising concerns about inflation."
]

## Step 5: Run and Display Results

Loop through each news article, call summarize_news(), and collect
all results into a list of dictionaries.

The list is then converted into a pandas DataFrame with six columns:
  News | Summary | Key Points | Insight | Sentiment | Tags

styled_df applies formatting so long text wraps cleanly within each cell,
making the table easy to read directly in the notebook.

In [ ]:
all_news_data = []
for news in news_list:
    result_dict = summarize_news(news)
    all_news_data.append(result_dict)


df = pd.DataFrame(all_news_data)
df = df[["News", "Summary", "Key Points", "Insight", "Sentiment", "Tags"]]

styled_df = df.style.set_properties(**{
    'white-space': 'pre-wrap',
    'text-align': 'left',
    'vertical-align': 'top'
})

styled_df